In [12]:
import sqlite3
from pathlib import Path
import pandas as pd

db_path = Path("/Users/chowingchan/Desktop/Project/AI-Analytics-Copilot/Competitive-Intelligence-Internal-Analytics-System/data/database/thelook_ecommerce.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

## mart_order_summary

In [13]:
cursor.execute("DROP VIEW IF EXISTS mart_order_summary;")

cursor.execute("""
CREATE VIEW mart_order_summary AS
SELECT
    o.order_id,
    o.user_id,
    o.status,
    o.created_at AS order_created_at,
    o.returned_at AS order_returned_at,
    o.shipped_at AS order_shipped_at,
    o.delivered_at AS order_delivered_at,

    COUNT(oi.id) AS item_count,
    SUM(oi.sale_price) AS order_revenue,
    SUM(p.cost) AS order_cost,
    SUM(oi.sale_price - p.cost) AS gross_profit,

    CASE 
        WHEN SUM(oi.sale_price) > 0 
        THEN ROUND(SUM(oi.sale_price - p.cost) * 1.0 / SUM(oi.sale_price), 4)
        ELSE NULL
    END AS gross_margin

FROM orders o
LEFT JOIN order_items oi
    ON o.order_id = oi.order_id
LEFT JOIN products p
    ON oi.product_id = p.id
GROUP BY
    o.order_id,
    o.user_id,
    o.status,
    o.created_at,
    o.returned_at,
    o.shipped_at,
    o.delivered_at;
""")

conn.commit()

In [14]:
#test

pd.read_sql("""
SELECT *
FROM mart_order_summary
LIMIT 10;
""", conn)

,order_id,user_id,status,order_created_at,order_returned_at,order_shipped_at,order_delivered_at,item_count,order_revenue,order_cost,gross_profit,gross_margin
0,1,1,Returned,2019-06-04 10:10:00+00:00,2019-06-08 20:53:00+00:00,2019-06-05 17:14:00+00:00,2019-06-08 15:55:00+00:00,2,63.990002,26.723791,37.266211,0.5824
1,2,2,Processing,2022-05-23 14:42:00+00:00,None,None,None,1,58.990002,26.840451,32.149551,0.5450
2,3,3,Complete,2020-06-20 15:09:00+00:00,None,2020-06-21 12:14:00+00:00,2020-06-24 17:28:00+00:00,3,93.490002,47.776461,45.713541,0.4890
3,4,3,Complete,2021-08-23 15:09:00+00:00,None,2021-08-26 07:58:00+00:00,2021-08-26 14:46:00+00:00,3,85.640001,41.676781,43.963221,0.5133
4,5,3,Shipped,2021-07-01 15:09:00+00:00,None,2021-07-03 11:21:00+00:00,None,1,279.000000,122.201999,156.798001,0.5620
5,6,4,Processing,2021-06-16 15:53:00+00:00,None,None,None,2,67.990000,33.970460,34.019540,0.5004
6,7,4,Cancelled,2022-05-08 15:53:00+00:00,None,None,None,3,364.479996,170.846078,193.633918,0.5313
7,8,5,Processing,2021-07-29 11:17:00+00:00,None,None,None,1,24.000000,10.536000,13.464000,0.5610
8,9,5,Returned,2021-09-20 11:17:00+00:00,2021-09-22 03:09:00+00:00,2021-09-20 13:33:00+00:00,2021-09-20 19:34:00+00:00,1,40.000000,16.960000,23.040000,0.5760
9,10,5,Shipped,2021-10-11 11:17:00+00:00,None,2021-10-13 02:04:00+00:00,None,1,29.990000,15.564810,14.425190,0.4810


In [15]:
#check grain

pd.read_sql("""
SELECT 
    COUNT(*) AS row_count,
    COUNT(DISTINCT order_id) AS distinct_orders
FROM mart_order_summary;
""", conn)

,row_count,distinct_orders
0,124923,124923


In [16]:
#check revenue

pd.read_sql("""
SELECT 
    status,
    COUNT(*) AS order_count,
    ROUND(SUM(order_revenue), 2) AS total_revenue,
    ROUND(AVG(order_revenue), 2) AS avg_order_value
FROM mart_order_summary
GROUP BY status
ORDER BY total_revenue DESC;
""", conn)

,status,order_count,total_revenue,avg_order_value
0,Shipped,37654,3251045.56,86.34
1,Complete,31028,2674091.08,86.18
2,Processing,24948,2142501.88,85.88
3,Cancelled,18723,1604482.06,85.70
4,Returned,12570,1086759.54,86.46


## mart_user_summary

In [17]:
cursor.execute("DROP VIEW IF EXISTS mart_user_summary;")

cursor.execute("""
CREATE VIEW mart_user_summary AS

WITH first_order AS (
    SELECT
        user_id,
        MIN(created_at) AS first_order_date
    FROM orders
    GROUP BY user_id
),

dataset_start AS (
    SELECT
        MIN(created_at) AS min_dataset_date
    FROM orders
),

rfm_base AS (
    SELECT
        o.user_id,

        MAX(o.created_at) AS last_order_date,

        COUNT(DISTINCT o.order_id) AS frequency,

        SUM(m.order_revenue) AS monetary

    FROM orders o
    LEFT JOIN mart_order_summary m
        ON o.order_id = m.order_id
    GROUP BY o.user_id
)

SELECT
    u.id AS user_id,
    u.gender,
    u.age,
    u.country,
    u.traffic_source,

    f.first_order_date,

    r.last_order_date,

    CAST(
        julianday(
            (SELECT MAX(created_at) FROM orders)
        ) - julianday(r.last_order_date)
    AS INTEGER) AS recency,

    r.frequency,

    ROUND(r.monetary, 2) AS monetary,
    
    CAST(
    julianday(
        (SELECT MAX(created_at) FROM orders)
        ) - julianday(f.first_order_date)
    AS INTEGER) AS days_since_first_order,

    CASE
        WHEN julianday(f.first_order_date)
             <= julianday(d.min_dataset_date) + 30
        THEN 'New'
        ELSE 'Existing'
    END AS customer_type

FROM users u

LEFT JOIN first_order f
    ON u.id = f.user_id

LEFT JOIN rfm_base r
    ON u.id = r.user_id

CROSS JOIN dataset_start d;
""")

conn.commit()

In [18]:
# Test mart_user_summary

pd.read_sql("""
SELECT *
FROM mart_user_summary
LIMIT 10;
""", conn)

,user_id,gender,age,country,traffic_source,first_order_date,last_order_date,recency,frequency,monetary,days_since_first_order,customer_type
0,44262,M,48,Japan,Facebook,2021-11-15 14:39:00+00:00,2021-11-15 14:39:00+00:00,205.0,1.0,80.00,205.0,Existing
1,61852,M,21,Japan,Search,2022-05-19 13:00:00+00:00,2022-05-19 13:00:00+00:00,20.0,1.0,40.00,20.0,Existing
2,82418,F,29,Japan,Search,2019-11-16 07:29:00+00:00,2021-06-16 07:29:00+00:00,357.0,2.0,86.99,935.0,Existing
3,23274,M,53,Brasil,Search,2021-05-06 14:39:00+00:00,2021-05-06 14:39:00+00:00,398.0,1.0,89.00,398.0,Existing
4,30022,F,23,Brasil,Email,2022-03-16 07:30:00+00:00,2022-03-16 07:30:00+00:00,84.0,1.0,47.00,84.0,Existing
5,35215,F,20,Brasil,Display,None,None,NaN,NaN,NaN,NaN,Existing
6,51769,M,14,Brasil,Search,2021-01-27 02:22:00+00:00,2021-01-27 02:22:00+00:00,497.0,1.0,25.00,497.0,Existing
7,64100,M,17,Brasil,Search,None,None,NaN,NaN,NaN,NaN,Existing
8,65914,F,51,Brasil,Facebook,2021-10-01 05:02:00+00:00,2021-10-01 05:02:00+00:00,250.0,1.0,37.45,250.0,Existing
9,67644,M,33,Brasil,Search,2020-08-22 10:57:00+00:00,2021-12-11 10:57:00+00:00,179.0,2.0,95.97,655.0,Existing


In [19]:
# test grain
pd.read_sql("""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT user_id) AS distinct_users
FROM mart_user_summary;
""", conn)

,row_count,distinct_users
0,100000,100000


In [20]:
# check customer type:
pd.read_sql("""
SELECT
    customer_type,
    COUNT(*) AS user_count
FROM mart_user_summary
GROUP BY customer_type;
""", conn)

,customer_type,user_count
0,Existing,99943
1,New,57


In [21]:
# check RFM ranges:
pd.read_sql("""
SELECT
    MIN(recency) AS min_recency,
    MAX(recency) AS max_recency,
    MIN(frequency) AS min_frequency,
    MAX(frequency) AS max_frequency,
    MIN(monetary) AS min_monetary,
    MAX(monetary) AS max_monetary
FROM mart_user_summary;
""", conn)

,min_recency,max_recency,min_frequency,max_frequency,min_monetary,max_monetary
0,0,1243,1,4,0.49,1753.32


In [22]:
pd.read_sql("""
SELECT
    frequency,
    COUNT(*) AS user_count
FROM mart_user_summary
GROUP BY frequency
ORDER BY frequency;
""", conn)

,frequency,user_count
0,NaN,20106
1,1.0,49923
2,2.0,19920
3,3.0,5044
4,4.0,5007


## mart_user_segment ( RFM Segmentation )

| Segment            | Logic                       |
| ------------------ | --------------------------- |
| Potential Users    | no orders                   |
| New Customers      | first 30 days               |
| One-time Buyers    | frequency = 1               |
| High Value Active  | recent + high monetary      |
| At Risk High Value | old recency + high monetary |
| Inactive Users     | recency > 180               |
| Regular Customers  | everyone else               |


| segment            | meaning                   |
| ------------------ | ------------------------- |
| Potential Users    | traffic but no conversion |
| New Customers      | recently converted        |
| One-time Buyers    | low retention             |
| High Value Active  | strong customers          |
| At Risk High Value | churn risk                |
| Inactive Users     | dormant customers         |


| user action    | creates user row? |
| -------------- | ----------------- |
| sign up        | YES               |
| browse website | maybe             |
| add to cart    | maybe             |
| checkout       | not necessarily   |
| purchase       | YES               |


In [23]:
segment_df = pd.read_sql("""
SELECT
    user_id,
    customer_type,
    recency,
    frequency,
    monetary
FROM mart_user_summary
""", conn)

In [24]:
#handle no-order users
segment_df['frequency'] = segment_df['frequency'].fillna(0)
segment_df['monetary'] = segment_df['monetary'].fillna(0)

In [25]:
#top 25% spenders
high_monetary_threshold = segment_df['monetary'].quantile(0.75)

print(high_monetary_threshold)

150.4925


In [26]:
def assign_segment(row):

    # no orders
    if row['frequency'] == 0:
        return "Prospect"
    
    # new customers
    elif row['customer_type'] == 'New':
        return "New Customers"
    
    # at risk high value
    elif (
        row['recency'] > 180
        and row['monetary'] >= high_monetary_threshold
    ):
        return "At Risk High Value"
    
    elif row['recency'] > 180:
        return "Inactive Users"

    # one-time buyers
    elif row['frequency'] == 1:
        return "One-time Buyers"

    # high value active
    elif (
        row['recency'] <= 30
        and row['monetary'] >= high_monetary_threshold
    ):
        return "High Value Active"

    else:
        return "Regular Customers"

In [27]:
segment_df['customer_segment'] = segment_df.apply(
    assign_segment,
    axis=1
)

In [28]:
#check distribution
segment_df['customer_segment'].value_counts()

customer_segment
Inactive Users        28592
One-time Buyers       21934
Prospect              20106
Regular Customers     15458
At Risk High Value    10320
High Value Active      3533
New Customers            57
Name: count, dtype: int64

In [30]:
#sample check

pd.read_sql("""
SELECT
    u.id,
    u.first_name,
    u.last_name,
    u.email,
    e.event_type,
    e.created_at
FROM users u
LEFT JOIN orders o
    ON u.id = o.user_id
LEFT JOIN events e
    ON u.id = e.user_id
LIMIT 20;
""", conn)

,id,first_name,last_name,email,event_type,created_at
0,44262,Michael,Sanchez,michaelsanchez@example.net,home,2021-11-15 12:02:56+00:00
1,44262,Michael,Sanchez,michaelsanchez@example.net,department,2021-11-15 12:05:01+00:00
2,44262,Michael,Sanchez,michaelsanchez@example.net,product,2021-11-15 12:06:44+00:00
3,44262,Michael,Sanchez,michaelsanchez@example.net,cart,2021-11-15 12:06:47+00:00
4,44262,Michael,Sanchez,michaelsanchez@example.net,purchase,2021-11-15 12:09:33+00:00
5,61852,David,Watson,davidwatson@example.org,home,2022-05-19 09:07:59+00:00
6,61852,David,Watson,davidwatson@example.org,department,2022-05-19 09:09:44+00:00
7,61852,David,Watson,davidwatson@example.org,product,2022-05-19 09:10:52+00:00
8,61852,David,Watson,davidwatson@example.org,cart,2022-05-19 09:11:12+00:00
9,61852,David,Watson,davidwatson@example.org,purchase,2022-05-19 09:12:04+00:00


In [31]:
segment_df.to_sql(
    "mart_user_segment",
    conn,
    if_exists="replace",
    index=False
)

100000

## Possible funnel analytics

| segment            | meaning                   |
| ------------------ | ------------------------- |
| Potential Users    | traffic but no conversion |
| New Customers      | recently converted        |
| One-time Buyers    | low retention             |
| High Value Active  | strong customers          |
| At Risk High Value | churn risk                |
| Inactive Users     | dormant customers         |


In [32]:
pd.read_sql("""
SELECT
    COUNT(*) AS total_events,
    COUNT(user_id) AS non_null_user_id,
    COUNT(DISTINCT user_id) AS distinct_user_ids
FROM events;
""", conn)

,total_events,non_null_user_id,distinct_user_ids
0,2420738,1296673,79893


In [33]:
pd.read_sql("""
SELECT *
FROM events
LIMIT 10;
""", conn)

,id,user_id,sequence_number,session_id,created_at,ip_address,city,state,postal_code,browser,traffic_source,uri,event_type
0,551,34.0,12,3c06e3f2-e8ed-4eb6-b24c-ccec40b8c2e7,2019-08-16 06:37:53+00:00,106.89.165.48,Kansas City,Kansas,66102,Chrome,Email,/cart,cart
1,1730,106.0,6,6d550704-9304-48df-990d-01fdbb676694,2021-02-22 05:56:40+00:00,103.12.134.61,New York,New York,11372,Chrome,Email,/cart,cart
2,2093,136.0,3,88457dec-f879-4f47-9585-98067ff94d75,2020-01-15 01:35:00+00:00,209.79.4.205,Beijing,Tianjin,300456,Firefox,Email,/cart,cart
3,2190,138.0,3,2729a4a5-1177-4a13-8419-f5f79f350876,2021-03-02 09:15:42+00:00,56.27.73.26,Quanzhou,Liaoning,110041,Chrome,Email,/cart,cart
4,2882,186.0,4,0933c284-960e-4f9b-9950-82e2d831c8e4,2020-01-19 00:54:42+00:00,45.195.175.235,Currais Novos,Rio Grande do Norte,59380-000,Safari,Email,/cart,cart
5,3460,223.0,12,7f2badab-2fd7-4fc2-81f7-680d38327919,2021-04-01 07:38:52+00:00,160.227.172.135,Terrassa,Cataluña,8224,Safari,Email,/cart,cart
6,4267,280.0,4,4ec2d8d5-8fb0-4874-b5b1-6fd65cb2096c,2021-03-28 14:03:46+00:00,137.234.165.61,Ayamonte,Andalucía,21400,Firefox,Email,/cart,cart
7,4301,284.0,4,6cfe802b-818b-4eaa-a700-32a7971af065,2022-01-08 08:45:56+00:00,92.149.131.23,Miami,Florida,33133,Chrome,Email,/cart,cart
8,5399,386.0,4,575e190c-bc63-4194-ab93-723fca06bfe1,2022-01-05 13:53:02+00:00,215.1.178.31,Shaoxing,Beijing,100123,Safari,Email,/cart,cart
9,6204,439.0,4,ce582f32-848c-4ad4-805e-b137bed82b6e,2020-12-24 05:45:01+00:00,61.64.118.74,Auburn,California,95603,Chrome,Email,/cart,cart


## mart_product_sales

In [34]:
cursor.execute("DROP VIEW IF EXISTS mart_product_sales;")

cursor.execute("""
CREATE VIEW mart_product_sales AS
SELECT
    DATE(o.created_at) AS order_date,
    p.id AS product_id,
    p.name AS product_name,
    p.category,
    p.brand,
    p.department,

    COUNT(DISTINCT oi.order_id) AS total_orders,
    COUNT(oi.id) AS units_sold,
    SUM(oi.sale_price) AS gross_sales,
    AVG(oi.sale_price) AS avg_selling_price

FROM order_items oi
JOIN orders o
    ON oi.order_id = o.order_id
JOIN products p
    ON oi.product_id = p.id

WHERE oi.status = 'Complete'

GROUP BY
    DATE(o.created_at),
    p.id,
    p.name,
    p.category,
    p.brand,
    p.department;
""")

conn.commit()

In [35]:
# check product row count

pd.read_sql("""
    SELECT order_date, product_id, COUNT(*)
    FROM mart_product_sales
    GROUP BY order_date, product_id
    HAVING COUNT(*) > 1;
""", conn)

,order_date,product_id,COUNT(*)


## table dict

| column              | meaning                               |
| ------------------- | ------------------------------------- |
| `order_date`        | sales date / grain date               |
| `product_id`        | product key                           |
| `product_name`      | product name                          |
| `category`          | product category                      |
| `brand`             | brand                                 |
| `department`        | department                            |
| `total_orders`      | distinct orders                       |
| `units_sold`        | sold items count                      |
| `gross_sales`       | total sales amount                    |
| `avg_selling_price` | average sale price                    |
| `returned_units`    | returned item count, optional         |
| `return_rate`       | returned_units / units_sold, optional |


## mart_daily_sales

Excluded completed order_items with null created_at from mart_daily_sales because the mart grain requires a valid order_date. These rows are logged as data quality exceptions.

In [36]:
cursor.execute("DROP VIEW IF EXISTS mart_daily_sales;")

cursor.execute("""
CREATE VIEW mart_daily_sales AS
SELECT
    DATE(oi.created_at) AS order_date,

    COUNT(DISTINCT oi.order_id) AS total_orders,
    COUNT(oi.id) AS units_sold,
    COUNT(DISTINCT o.user_id) AS active_customers,

    SUM(oi.sale_price) AS gross_sales,
    AVG(oi.sale_price) AS avg_item_price,
    SUM(oi.sale_price) * 1.0 / COUNT(DISTINCT oi.order_id) AS avg_order_value

FROM order_items oi
JOIN orders o
    ON oi.order_id = o.order_id

WHERE oi.status = 'Complete'
    AND oi.created_at IS NOT NULL

GROUP BY
    DATE(oi.created_at);
""")

conn.commit()

In [37]:
# check row count

pd.read_sql("""
    SELECT COUNT(*) 
    FROM mart_daily_sales;
""", conn)

,COUNT(*)
0,1215


In [38]:
# check grain

pd.read_sql("""
    SELECT order_date, COUNT(*)
    FROM mart_daily_sales
    GROUP BY order_date
    HAVING COUNT(*) > 1;
""", conn)

,order_date,COUNT(*)


In [39]:
# check revenue

pd.read_sql("""
    SELECT SUM(gross_sales)
    FROM mart_daily_sales;
""", conn)

,SUM(gross_sales)
0,2.612365e+06


In [40]:
pd.read_sql("""
    SELECT SUM(sale_price)
    FROM order_items
    WHERE status = 'Complete';
""", conn)

,SUM(sale_price)
0,2.674091e+06


In [41]:
pd.read_sql("""
    SELECT
        MIN(DATE(created_at)) AS min_date,
        MAX(DATE(created_at)) AS max_date,
        COUNT(DISTINCT DATE(created_at)) AS active_sales_days
    FROM order_items
    WHERE status = 'Complete';
""", conn)

,min_date,max_date,active_sales_days
0,2019-01-12,2022-06-12,1215


In [44]:
pd.read_sql("""
    SELECT *
    FROM mart_daily_sales
    WHERE order_date IS NULL;
""", conn)

,order_date,total_orders,units_sold,active_customers,gross_sales,avg_item_price,avg_order_value


In [43]:
# check database

pd.read_sql("""
SELECT name, type
FROM sqlite_master
ORDER BY type, name;
""", conn)

,name,type
0,distribution_centers,table
1,events,table
2,inventory_items,table
3,mart_user_segment,table
4,order_items,table
5,orders,table
6,products,table
7,users,table
8,mart_daily_sales,view
9,mart_order_summary,view
